In [ ]:
import torch
from torch.nn import functional as F

# 데이터 3개의 클래스별 logit
logits = torch.tensor(
    [
        [4.7, 12.1, 3.2],
        [5.7, 8.9, 1.2],
        [11.0, 4.0, 2.3],
    ]
)

# 각 데이터의 정답 클래스
labels = torch.tensor(
    [
        0,
        1,
        2,
    ]
)

print("Logits:")
print(logits)
print("Logits shape:", logits.shape)

print("\nLabels:")
print(labels)
print("Labels shape:", labels.shape)

Logits:
tensor([[ 4.7000, 12.1000,  3.2000],
        [ 5.7000,  8.9000,  1.2000],
        [11.0000,  4.0000,  2.3000]])
Logits shape: torch.Size([3, 3])

Labels:
tensor([0, 1, 2])
Labels shape: torch.Size([3])


In [ ]:
large_logits = torch.tensor(
    [
        [1000.0, 999.0, 998.0],
    ]
)

# 수학적 정의를 그대로 계산하면 overflow가 발생한다.
naive_exponentials = torch.exp(
    large_logits,
)

naive_probabilities = naive_exponentials / naive_exponentials.sum(
    dim=-1,
    keepdim=True,
)


# 행별 최댓값을 빼면 가장 큰 logit이 0이 된다.
shifted_logits = (
    large_logits
    - large_logits.max(
        dim=-1,
        keepdim=True,
    ).values
)

stable_probabilities = torch.softmax(
    shifted_logits,
    dim=-1,
)

print("Naive probabilities:")
print(naive_probabilities)

print("\nShifted logits:")
print(shifted_logits)

print("\nStable probabilities:")
print(stable_probabilities)

Naive probabilities:
tensor([[nan, nan, nan]])

Shifted logits:
tensor([[ 0., -1., -2.]])

Stable probabilities:
tensor([[0.6652, 0.2447, 0.0900]])


In [ ]:
def cross_entropy_loss(
    logits,
    labels,
    averaged=True,
):
    """Logit을 직접 받아 cross-entropy를 계산"""

    # 마지막 차원은 클래스 차원으로 유지
    logits = logits.reshape(
        -1,
        logits.shape[-1],
    )

    # 정답 클래스 번호를 1차원으로 펼치기
    labels = labels.reshape(-1)

    return F.cross_entropy(
        logits,
        labels,
        reduction=("mean" if averaged else "none"),
    )


per_example_loss = cross_entropy_loss(
    logits,
    labels,
    averaged=False,
)

mean_loss = cross_entropy_loss(
    logits,
    labels,
    averaged=True,
)

print("Per-example loss:")
print(per_example_loss)
print("Shape:", per_example_loss.shape)

print("\nMean loss:")
print(mean_loss)
print("Shape:", mean_loss.shape)

Per-example loss:
tensor([7.4007, 0.0404, 8.7011])
Shape: torch.Size([3])

Mean loss:
tensor(5.3807)
Shape: torch.Size([])


In [ ]:
log_probabilities = F.log_softmax(
    logits,
    dim=-1,
)

batch_indices = torch.arange(
    labels.shape[0],
)

manual_loss = -log_probabilities[
    batch_indices,
    labels,  # 정답 인덱스 번호 (ex: [0, 1, 2, 1, 0, 2...])
]

print("Log probabilities:")
print(log_probabilities)

print("\nManual loss:")
print(manual_loss)

print("\nF.cross_entropy loss:")
print(per_example_loss)

assert torch.allclose(
    manual_loss,
    per_example_loss,
)

Log probabilities:
tensor([[-7.4007e+00, -7.4740e-04, -8.9007e+00],
        [-3.2404e+00, -4.0388e-02, -7.7404e+00],
        [-1.0778e-03, -7.0011e+00, -8.7011e+00]])

Manual loss:
tensor([7.4007, 0.0404, 8.7011])

F.cross_entropy loss:
tensor([7.4007, 0.0404, 8.7011])


In [24]:
probabilities = torch.softmax(
    logits,
    dim=-1,
)

predictions = probabilities.argmax(
    dim=-1,
)

print("Probabilities:")
print(probabilities)

print("\nProbability sums:")
print(probabilities.sum(dim=-1))

print("\nPredictions:")
print(predictions)

Probabilities:
tensor([[6.1080e-04, 9.9925e-01, 1.3629e-04],
        [3.9149e-02, 9.6042e-01, 4.3490e-04],
        [9.9892e-01, 9.1090e-04, 1.6641e-04]])

Probability sums:
tensor([1.0000, 1.0000, 1.0000])

Predictions:
tensor([1, 1, 0])
